<a href="https://colab.research.google.com/github/Decoding-Data-Science/CommunityWorkshops/blob/main/bootcampaug26/HFSpaces-ddsenterchatbot-26jul.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pinecone Vector Store

If you're opening this Notebook on colab, you will probably need to install LlamaIndex 🦙.

In [ ]:
%pip install llama-index llama-index-vector-stores-pinecone llama-index-readers-file

In [2]:
import logging
import sys
import os

logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

#### Creating a Pinecone Index

In [5]:
from pinecone import Pinecone, ServerlessSpec

In [3]:
import openai
from google.colab import userdata

# Retrieve the OpenAI API key from Google Colab secrets
openai.api_key = userdata.get('openai')

In [6]:
import openai
from google.colab import userdata

# Retrieve the OpenAI API key from Google Colab secrets
api_key = userdata.get('PINECONE_API_KEY')

#api_key = os.environ["PINECONE_API_KEY"]

pc = Pinecone(api_key=api_key)

In [7]:
#delete if needed ( first time no need to delete) DOn't RUN
pc.delete_index("quickstart")

In [8]:
# dimensions are for text-embedding-ada-002

pc.create_index(
    name="quickstart",
    dimension=1536,
    metric="euclidean",
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)


IndexModel(name='quickstart', metric='euclidean', status=IndexStatus(ready=True, state='Ready'), spec=IndexSpec(serverless=ServerlessSpecInfo(cloud='aws', region='us-east-1', read_capacity={'mode': 'OnDemand', 'status': {'state': 'Ready', 'current_shards': None, 'current_replicas': None}}, source_collection=None, schema=None), pod=None, byoc=None), host='https://quickstart-05hy6j9.svc.aped-4627-b74a.pinecone.io', private_host=None, vector_type='dense', dimension=1536, deletion_protection='disabled', tags=None, embed=None, created_at=None)

In [9]:
pinecone_index = pc.Index("quickstart")

#### Load documents, build the PineconeVectorStore and VectorStoreIndex

In [10]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.vector_stores.pinecone import PineconeVectorStore
from IPython.display import Markdown, display

Download Data

In [11]:
#2. upload DDS HR enterprise files directly from system
import os
from google.colab import files
import shutil

# Define the target directory
target_directory = './data/'

# Create the target directory if it doesn't exist
os.makedirs(target_directory, exist_ok=True)

print("Please select the file you want to upload:")
uploaded = files.upload()

for filename in uploaded.keys():
  # Define the destination path for the uploaded file
  destination_path = os.path.join(target_directory, filename)

  # Move the uploaded file from the current directory to the target directory
  shutil.move(filename, destination_path)
  print(f"File '{filename}' successfully uploaded and moved to '{target_directory}'")


Please select the file you want to upload:


Saving DDS_Employee_Handbook_Synthetic_v1.pdf to DDS_Employee_Handbook_Synthetic_v1.pdf
Saving DDS_HR_FAQ_Synthetic_v1.pdf to DDS_HR_FAQ_Synthetic_v1.pdf
Saving DDS_Leave_Policy_v2.pdf to DDS_Leave_Policy_v2.pdf
Saving DDS_Remote_Work_Policy_Synthetic_v1.pdf to DDS_Remote_Work_Policy_Synthetic_v1.pdf
File 'DDS_Employee_Handbook_Synthetic_v1.pdf' successfully uploaded and moved to './data/'
File 'DDS_HR_FAQ_Synthetic_v1.pdf' successfully uploaded and moved to './data/'
File 'DDS_Leave_Policy_v2.pdf' successfully uploaded and moved to './data/'
File 'DDS_Remote_Work_Policy_Synthetic_v1.pdf' successfully uploaded and moved to './data/'


In [12]:
from llama_index.core import SimpleDirectoryReader
from llama_index.readers.file import PDFReader

documents = SimpleDirectoryReader(
    input_dir="data",
    required_exts=[".pdf"],
    file_extractor={".pdf": PDFReader()}
).load_data()

print(len(documents))
print(documents[0].text[:1000])

13
DDS Employee Handbook (Synthetic) v1
Effective date: March 03, 2026  Dubai (GST)
Note: This document is a synthetic, training-friendly employee handbook for demos, onboarding
simulations, and HR-policy chatbot prototypes. It is not legal advice and must be reviewed by qualified
counsel before any real-world use.
1. Welcome to Decoding Data Science (DDS)
DDS is a Dubai-based academy, consulting practice, and community focused on data science, AI, and
applied generative AI. We operate with a global mindset and a high trust culture—shipping practical
outcomes while supporting each other.
This handbook explains workplace expectations, benefits, and policies. If any local law conflicts with
this handbook, applicable law prevails.
2. Company Values & Ways of Working
 Build with clarity: define the user, problem, inputs/outputs, and definition of done.
 Bias for action: ship small, iterate fast, measure outcomes.
 Respect and inclusion: disagreement is allowed; disrespect is not.
 Dat

In [13]:
documents

[Document(id_='121ce2d8-0fab-43f1-a0a3-dad6f12ad028', embedding=None, metadata={'page_label': '1', 'file_name': 'DDS_Employee_Handbook_Synthetic_v1.pdf', 'file_path': '/content/data/DDS_Employee_Handbook_Synthetic_v1.pdf', 'file_type': 'application/pdf', 'file_size': 9665, 'creation_date': '2026-08-30', 'last_modified_date': '2026-08-30'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='DDS Employee Handbook (Synthetic) v1\nEffective date: March 03, 2026 \x7f Dubai (GST)\nNote: This document is a synthetic, training-friendly employee handbook for demos, onboarding\nsimulations, and HR-policy chatbot prototypes. It is not legal advice and m

In [14]:
# initialize without metadata filter
from llama_index.core import StorageContext

vector_store = PineconeVectorStore(pinecone_index=pinecone_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context
)

Upserting:   0%|          | 0/1 [00:00<?, ?batch/s]

#### Query Index

May take a minute or so for the index to be ready!

In [19]:
# set Logging to DEBUG for more detailed outputs
query_engine = index.as_query_engine()
response = query_engine.query("what is standard working hours in decoding data  science")
display(Markdown(f"<b>{response}</b>"))

<b>The standard office hours at Decoding Data Science (DDS) are from 9:00 AM to 6:00 PM, Monday to Friday.</b>

In [20]:
pip install gradio

In [24]:
import gradio as gr

def greet(name):
    return "Hello " + name + "!"

demo = gr.Interface(fn=greet, inputs="text", outputs="text")
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/08/30 11:48:37 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: connect: connection refused


<IPython.core.display.Javascript object>

In [21]:
# If needed in Colab, install first:
# !pip install -U gradio pinecone llama-index llama-index-vector-stores-pinecone llama-index-readers-file pypdf
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext, Settings
# --- Imports ---
import logging
import sys
import gradio as gr

from pinecone import Pinecone, ServerlessSpec
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext , Settings
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.readers.file import PDFReader
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
# --- Logging ---
logging.basicConfig(stream=sys.stdout, level=logging.INFO)


Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0.2)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-ada-002")
Settings.chunk_size = 600
Settings.chunk_overlap = 200

# Define a system prompt
system_prompt = '''
You are AYesha, the Decoding Data Science (DDS) Enterprise HR Chatbot. Answer questions exclusively using the attached DDS HR Handbook. Base all responses on the most up-to-date information available in the handbook. Only respond to queries directly related to DDS HR policies as outlined in the handbook.

- If a question pertains to topics outside DDS HR policies, respond politely, clarifying that you are a human resources bot and only answer DDS HR questions.
- For questions you cannot answer (e.g., requests for old policies, salary details, or confidential information), politely decline and direct the user to email connect@decodingdatascience.com.
- Never answer questions about anything outside of your scope.
- Persist in following these constraints for any follow-up questions.
- Before answering, carefully check that the information and query are within the allowed scope. Follow chain-of-thought reasoning:
  1. First, reason step-by-step whether the question is covered in the current handbook and is within HR.
  2. Only after confirming, produce a final answer.

Format answers as concise, professional responses. Do not wrap answers in code blocks or any special formatting.

Output requirements:
- For allowed HR questions, answer concisely based only on the latest DDS HR handbook information.
- For forbidden topics, output: “I’m sorry, I can only answer questions about the latest DDS HR policies. For confidential or other queries, please email connect@decodingdatascience.com.”


**Example 1**
User: What is the leave encashment policy at DDS?
Reasoning: This is an HR policy question found in the latest handbook.
Final Answer: [Provide answer summarized from the latest handbook’s section on leave encashment]

**Example 2**
User: Can you tell me the salary range for Data Scientists?
Reasoning: Salary details are confidential and not shared by this bot.
Final Answer: I’m sorry, I can only answer questions about the latest DDS HR policies. For confidential or other queries, please email connect@decodingdatascience.com.

**Example 3**
User: Can you explain what DDS does as a company overall?
Reasoning: This is not an HR question, so it cannot be answered.
Final Answer: I’m sorry, I only answer DDS HR policy questions as outlined in the handbook.

(Real-world examples should be longer and use precise wording from the handbook where appropriate.)

**Important instructions:**
- Only answer questions directly supported by the latest DDS HR handbook.
- Decline politely and redirect to the provided email address for any questions outside scope or for confidential information.
- Always reason before concluding. Only present the answer after checking scope and source.

Remember: As AYesha, the DDS HR Enterprise Chatbot, you must never provide information outside authorized HR handbook content and always respond respectfully according to these constraints.

'''


# --- Load API Key from Colab Secrets ---
# In Colab: left panel -> Secrets -> add secret named PINECONE_API_KEY
PINECONE_API_KEY = userdata.get("PINECONE_API_KEY")


# --- Initialize Pinecone ---
pc = Pinecone(api_key=PINECONE_API_KEY)
index_name = "quickstart"
dimension = 1536

# --- Delete index if it already exists (optional) ---
existing_indexes = [idx["name"] for idx in pc.list_indexes()]

if index_name in existing_indexes:
    pc.delete_index(index_name)

# --- Create Pinecone index ---
pc.create_index(
    name=index_name,
    dimension=dimension,
    metric="euclidean",
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)

pinecone_index = pc.Index(index_name)

# --- Load PDF documents from folder ---
documents = SimpleDirectoryReader(
    input_dir="data",
    required_exts=[".pdf"],
    file_extractor={".pdf": PDFReader()}
).load_data()

if not documents:
    raise ValueError("No PDF documents were loaded from the 'data' folder.")

# --- Create Vector Index ---
vector_store = PineconeVectorStore(pinecone_index=pinecone_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context
)

# --- Query Engine ---
query_engine = index.as_query_engine(system_prompt=system_prompt)

# --- Gradio App ---
def query_doc(prompt):
    try:
        response = query_engine.query(prompt)
        return str(response)
    except Exception as e:
        return f"Error: {str(e)}"

gr.Interface(
    fn=query_doc,
    inputs=gr.Textbox(label="Ask a question about the document"),
    outputs=gr.Textbox(label="Answer"),
    title="DDS Enterprise HR Chatbot",
    description="Ask questions related to HR for latest Information."
).launch(share=True,debug=True)

Upserting:   0%|          | 0/1 [00:00<?, ?batch/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/08/30 11:34:00 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: connect: connection refused


<IPython.core.display.Javascript object>

In [ ]:
# for hugingface to create app.py
# !pip install -U gradio pinecone llama-index llama-index-vector-stores-pinecone llama-index-readers-file pypdf
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext, Settings
# --- Imports ---
import logging
import sys
import gradio as gr


from pinecone import Pinecone, ServerlessSpec
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext , Settings
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.readers.file import PDFReader
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
# --- Logging ---
logging.basicConfig(stream=sys.stdout, level=logging.INFO)

#load keys for huggingface
import os
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")


Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0.2)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-ada-002")
Settings.chunk_size = 600
Settings.chunk_overlap = 200

# Define a system prompt
system_prompt = '''
You are AYesha, the Decoding Data Science (DDS) Enterprise HR Chatbot. Answer questions exclusively using the attached DDS HR Handbook. Base all responses on the most up-to-date information available in the handbook. Only respond to queries directly related to DDS HR policies as outlined in the handbook.

- If a question pertains to topics outside DDS HR policies, respond politely, clarifying that you are a human resources bot and only answer DDS HR questions.
- For questions you cannot answer (e.g., requests for old policies, salary details, or confidential information), politely decline and direct the user to email connect@decodingdatascience.com.
- Never answer questions about anything outside of your scope.
- Persist in following these constraints for any follow-up questions.
- Before answering, carefully check that the information and query are within the allowed scope. Follow chain-of-thought reasoning:
  1. First, reason step-by-step whether the question is covered in the current handbook and is within HR.
  2. Only after confirming, produce a final answer.

Format answers as concise, professional responses. Do not wrap answers in code blocks or any special formatting.

Output requirements:
- For allowed HR questions, answer concisely based only on the latest DDS HR handbook information.
- For forbidden topics, output: “I’m sorry, I can only answer questions about the latest DDS HR policies. For confidential or other queries, please email connect@decodingdatascience.com.”


**Example 1**
User: What is the leave encashment policy at DDS?
Reasoning: This is an HR policy question found in the latest handbook.
Final Answer: [Provide answer summarized from the latest handbook’s section on leave encashment]

**Example 2**
User: Can you tell me the salary range for Data Scientists?
Reasoning: Salary details are confidential and not shared by this bot.
Final Answer: I’m sorry, I can only answer questions about the latest DDS HR policies. For confidential or other queries, please email connect@decodingdatascience.com.

**Example 3**
User: Can you explain what DDS does as a company overall?
Reasoning: This is not an HR question, so it cannot be answered.
Final Answer: I’m sorry, I only answer DDS HR policy questions as outlined in the handbook.

(Real-world examples should be longer and use precise wording from the handbook where appropriate.)

**Important instructions:**
- Only answer questions directly supported by the latest DDS HR handbook.
- Decline politely and redirect to the provided email address for any questions outside scope or for confidential information.
- Always reason before concluding. Only present the answer after checking scope and source.

Remember: As AYesha, the DDS HR Enterprise Chatbot, you must never provide information outside authorized HR handbook content and always respond respectfully according to these constraints.

'''




# --- Initialize Pinecone ---
pc = Pinecone(api_key=PINECONE_API_KEY)
index_name = "quickstart"
dimension = 1536

# --- Delete index if it already exists (optional) ---
existing_indexes = [idx["name"] for idx in pc.list_indexes()]

if index_name in existing_indexes:
    pc.delete_index(index_name)

# --- Create Pinecone index ---
pc.create_index(
    name=index_name,
    dimension=dimension,
    metric="euclidean",
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)

pinecone_index = pc.Index(index_name)

# --- Load PDF documents from folder ---
documents = SimpleDirectoryReader(
    input_dir="Data",
    required_exts=[".pdf"],
    file_extractor={".pdf": PDFReader()}
).load_data()

if not documents:
    raise ValueError("No PDF documents were loaded from the 'data' folder.")

# --- Create Vector Index ---
vector_store = PineconeVectorStore(pinecone_index=pinecone_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context
)

# --- Query Engine ---
query_engine = index.as_query_engine(system_prompt=system_prompt)

# --- Gradio App ---
def query_doc(prompt):
    try:
        response = query_engine.query(prompt)
        return str(response)
    except Exception as e:
        return f"Error: {str(e)}"

gr.Interface(
    fn=query_doc,
    inputs=gr.Textbox(label="Ask a question about the document"),
    outputs=gr.Textbox(label="Answer"),
    title="DDS Enterprise HR Chatbot",
    description="Ask questions related to HR for latest Information."
).launch(share=True)









In [ ]:
import logging
import sys
import gradio as gr
from pinecone import Pinecone, ServerlessSpec
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext , Settings
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.readers.file import PDFReader
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from google.colab import userdata # Import userdata for API keys
import os # Import os for environment variables

# --- Logging ---
logging.basicConfig(stream=sys.stdout, level=logging.INFO)

# --- Load API Key from Colab Secrets ---
# In Colab: left panel -> Secrets -> add secret named PINECONE_API_KEY and openai
PINECONE_API_KEY = userdata.get("PINECONE_API_KEY")
OPENAI_API_KEY = userdata.get("openai") # Get OpenAI key from secrets

# Ensure OpenAI API key is set for llama_index
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


# --- Initialize Pinecone ---
pc = Pinecone(api_key=PINECONE_API_KEY)
index_name = "quickstart"
dimension = 1536

# --- Delete index if it already exists (optional, for fresh runs) ---
existing_indexes = [idx["name"] for idx in pc.list_indexes()]

if index_name in existing_indexes:
    pc.delete_index(index_name)
    print(f"Deleted existing index: {index_name}")

# --- Create Pinecone index ---
pc.create_index(
    name=index_name,
    dimension=dimension,
    metric="euclidean",
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)
print(f"Created Pinecone index: {index_name}")

pinecone_index = pc.Index(index_name)

# --- Load PDF documents from folder ---
# Ensure 'data' directory exists and contains PDF files
documents = SimpleDirectoryReader(
    input_dir="data",
    required_exts=[".pdf"],
    file_extractor={".pdf": PDFReader()}
).load_data()

if not documents:
    raise ValueError("No PDF documents were loaded from the 'data' folder. Please ensure the 'data' folder exists and contains PDF files.")
print(f"Loaded {len(documents)} documents from 'data' folder.")

# --- Create Vector Index ---
vector_store = PineconeVectorStore(pinecone_index=pinecone_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context
)
print("VectorStoreIndex created.")

# --- Query Engine Settings and System Prompt ---
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0.2)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-ada-002")
Settings.chunk_size = 600
Settings.chunk_overlap = 200

system_prompt = '''
You are AYesha, the Decoding Data Science (DDS) Enterprise HR Chatbot. Answer questions exclusively using the attached DDS HR Handbook. Base all responses on the most up-to-date information available in the handbook. Only respond to queries directly related to DDS HR policies as outlined in the handbook.

- If a question pertains to topics outside DDS HR policies, respond politely, clarifying that you are a human resources bot and only answer DDS HR questions.
- For questions you cannot answer (e.g., requests for old policies, salary details, or confidential information), politely decline and direct the user to email connect@decodingdatascience.com.
- Never answer questions about anything outside of your scope.
- Persist in following these constraints for any follow-up questions.
- Before answering, carefully check that the information and query are within the allowed scope. Follow chain-of-thought reasoning:
  1. First, reason step-by-step whether the question is covered in the current handbook and is within HR.
  2. Only after confirming, produce a final answer.

Format answers as concise, professional responses. Do not wrap answers in code blocks or any special formatting.

Output requirements:
- For allowed HR questions, answer concisely based only on the latest DDS HR handbook information.
- For forbidden topics, output: “I’m sorry, I can only answer questions about the latest DDS HR policies. For confidential or other queries, please email connect@decodingdatascience.com.”


**Example 1**
User: What is the leave encashment policy at DDS?
Reasoning: This is an HR policy question found in the latest handbook.
Final Answer: [Provide answer summarized from the latest handbook’s section on leave encashment]

**Example 2**
User: Can you tell me the salary range for Data Scientists?
Reasoning: Salary details are confidential and not shared by this bot.
Final Answer: I’m sorry, I can only answer questions about the latest DDS HR policies. For confidential or other queries, please email connect@decodingdatascience.com.

**Example 3**
User: Can you explain what DDS does as a company overall?
Reasoning: This is not an HR question, so it cannot be answered.
Final Answer: I’m sorry, I only answer DDS HR policy questions as outlined in the handbook.

(Real-world examples should be longer and use precise wording from the handbook where appropriate.)

**Important instructions:**
- Only answer questions directly supported by the latest DDS HR handbook.
- Decline politely and redirect to the provided email address for any questions outside scope or for confidential information.
- Always reason before concluding. Only present the answer after checking scope and source.

Remember: As AYesha, the DDS HR Enterprise Chatbot, you must never provide information outside authorized HR handbook content and always respond respectfully according to these constraints.
'''

query_engine = index.as_query_engine(system_prompt=system_prompt)
print("Query engine initialized with system prompt.")

# --- Gradio App ---
# Original query_doc function - kept for core logic, wrapped by chatbot_response
def query_doc(prompt):
    try:
        response = query_engine.query(prompt)
        return str(response)
    except Exception as e:
        return f"Error: {str(e)}"

# Wrapper function for Gradio Chatbot to integrate with query_doc
def chatbot_response(message, history):
    # Call the original query_doc function
    bot_message = query_doc(message)
    # Append the user message and bot response to the history for display
    history.append([message, bot_message])
    return history

faq_questions = [
    "What are the standard working hours in DDS?",
    "What is the leave policy at DDS?",
    "How do I report harassment or discrimination?",
    "What are the company's values?",
    "What is the policy on remote work?",
    "What are the types of employment classification at DDS?"
]

with gr.Blocks(title="DDS Enterprise HR Chatbot") as demo: # Removed theme from here as per warning
    gr.Markdown(
        """
        <div style='text-align: center;'>
            <h1>DDS Enterprise HR Chatbot</h1>
            <p>Ask questions related to HR for the latest information from the DDS HR Handbook.</p>
        </div>
        """
    )
    with gr.Row():
        with gr.Column(scale=2):
            gr.Markdown("## Chat with AYesha")
            # Using gr.Chatbot for a more interactive chat history
            chatbot = gr.Chatbot(height=400, label="Chat History")
            msg = gr.Textbox(label="Your Question", placeholder="E.g., What is the annual leave policy?", lines=2)

            with gr.Row():
                # Clear button for both the message input and the chatbot history
                clear_button = gr.ClearButton([msg, chatbot])
                submit_button = gr.Button("Send", variant="primary")

            # Link text input and submit button to the chatbot_response function
            msg.submit(chatbot_response, inputs=[msg, chatbot], outputs=chatbot, concurrency_limit=None)
            submit_button.click(chatbot_response, inputs=[msg, chatbot], outputs=chatbot, concurrency_limit=None)
            # Clear the message input box after sending
            msg.submit(lambda: None, outputs=msg)
            submit_button.click(lambda: None, outputs=msg)


        with gr.Column(scale=1):
            gr.Markdown("## Frequently Asked Questions")
            for i, q in enumerate(faq_questions):
                with gr.Accordion(f"**Q{i+1}: {q}**", open=False):
                    gr.Markdown("Click the button below to ask the chatbot this question directly.")
                    faq_button = gr.Button("Ask Chatbot", elem_id=f"faq_btn_{i}")

                    # Chain of events for FAQ button:
                    # 1. Set the question text into the message input box
                    # 2. Call the chatbot_response function with the new message and current history
                    # 3. Clear the message input box
                    faq_button.click(
                        lambda q_text: q_text, # Function to just pass the question string
                        inputs=gr.State(q),    # Pass the specific FAQ question as state
                        outputs=msg            # Update the message input textbox
                    ).then(
                        chatbot_response,      # Then call chatbot_response
                        inputs=[msg, chatbot], # Using the updated message and current chatbot history
                        outputs=chatbot        # Update the chatbot history display
                    ).then(
                        lambda: None, outputs=msg # Clear the message input after the chatbot responds
                    )

# Launch the Gradio app with the theme and title parameters as per Gradio 6.0 best practices
demo.launch(share=True, theme=gr.themes.Soft())

Deleted existing index: quickstart
Created Pinecone index: quickstart
Loaded 13 documents from 'data' folder.


Upserted vectors:   0%|          | 0/15 [00:00<?, ?it/s]

VectorStoreIndex created.
Query engine initialized with system prompt.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4c458f9b54af1483cc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
#### This for hugging face space for zero gpu
##1. create requirements.txt
##2.add keys in HF secrets
##3. put this in app.py below
# --- Imports ---
import logging
import os
import sys

import gradio as gr
import spaces

from pinecone import Pinecone, ServerlessSpec
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    Settings,
    PromptTemplate,
)
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.readers.file import PDFReader
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# --- Logging ---
logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logger = logging.getLogger(__name__)

# --- Keys (set these in Space Settings -> Variables and secrets) ---
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")



# --- LlamaIndex global settings ---
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0.2)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-ada-002")
Settings.chunk_size = 600
Settings.chunk_overlap = 200

SYSTEM_PROMPT = """You are AYesha, the Decoding Data Science (DDS) Enterprise HR Chatbot. Answer questions exclusively using the attached DDS HR Handbook. Base all responses on the most up-to-date information available in the handbook. Only respond to queries directly related to DDS HR policies as outlined in the handbook.
Important instructions:
- Only answer questions directly supported by the latest DDS HR handbook.
- Decline politely and redirect to the provided email address for any questions outside scope or for confidential information.
- Always reason before concluding. Only present the answer after checking scope and source.
Remember: As AYesha, the DDS HR Enterprise Chatbot, you must never provide information outside authorized HR handbook content and always respond respectfully according to these constraints.
"""

# LlamaIndex's query engine doesn't accept a `system_prompt=` kwarg — that param only
# exists on as_chat_engine(). The equivalent for a query engine is a custom QA prompt
# template, which is what actually enforces AYesha's scope/persona.
QA_TEMPLATE = PromptTemplate(
    SYSTEM_PROMPT
    + "\n\nContext information is below.\n---------------------\n{context_str}\n---------------------\n"
    "Given the context information and not prior knowledge, answer the query.\n"
    "Query: {query_str}\nAnswer: "
)

# --- Initialize Pinecone ---
pc = Pinecone(api_key=PINECONE_API_KEY)
INDEX_NAME = "quickstart"
DIMENSION = 1536

existing_indexes = [idx["name"] for idx in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    # Only create the index the first time. Deleting + recreating on every
    # restart re-embeds every PDF (real API cost) and risks a race if two
    # cold starts overlap — so we only build the index when it doesn't exist yet.
    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSION,
        metric="euclidean",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

pinecone_index = pc.Index(INDEX_NAME)
vector_store = PineconeVectorStore(pinecone_index=pinecone_index)

# If the index is already populated (from a prior run), just attach to it.
# Otherwise load the PDFs and build it now.
stats = pinecone_index.describe_index_stats()
if stats.get("total_vector_count", 0) > 0:
    index = VectorStoreIndex.from_vector_store(vector_store)
else:
    documents = SimpleDirectoryReader(
        input_dir="Data",  # folder name is case-sensitive — must match exactly what you upload
        required_exts=[".pdf"],
        file_extractor={".pdf": PDFReader()},
    ).load_data()

    if not documents:
        raise ValueError(
            "No PDF documents were loaded from the 'Data' folder. "
            "Make sure a folder named exactly 'Data' with your PDF(s) is uploaded alongside app.py."
        )

    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)

query_engine = index.as_query_engine(text_qa_template=QA_TEMPLATE)


# --- Gradio App ---
@spaces.GPU
def query_doc(prompt):
    try:
        response = query_engine.query(prompt)
        return str(response)
    except Exception as e:
        logger.exception("Query failed")
        return f"Error: {str(e)}"


gr.Interface(
    fn=query_doc,
    inputs=gr.Textbox(label="Ask a question about the document"),
    outputs=gr.Textbox(label="Answer"),
    title="DDS Enterprise HR Chatbot — AYesha",
    description="Ask questions related to HR for latest information.",
).launch()